# 第5章  向量的点积（内积） —— 相似度的度量

> **本章目标**：掌握点积的两种等价定义，用余弦相似度构建推荐系统，建立全书最重要的AI连接：Transformer的Q·K注意力得分就是点积。
> **前置知识**：第3章（向量、shape）、第4章（向量加减数乘）


In [ ]:
import numpy as np
print('环境就绪')


---
## 5.1  代数定义 —— 对应位置相乘再求和

a·b = sum(a_i * b_i)。点积>0→夹角<90°，<0→夹角>90°，=0→垂直。

AI连接：PyTorch的F.linear(input,weight)内部每个元素都是一次点积。


In [ ]:
import numpy as np
a = np.array([2.0, 3.0, 1.0])
b = np.array([4.0, 1.0, 5.0])
dot_manual = sum(a[i]*b[i] for i in range(len(a)))
dot_np = np.dot(a, b)
dot_at = a @ b
print(f'manual={dot_manual}, np.dot={dot_np}, a@b={dot_at}')
print(f'all equal: {dot_manual==dot_np==dot_at}')

a_pos = np.array([3.0, 4.0])
b_pos = np.array([2.0, 1.0])
b_neg = np.array([-2.0, -1.0])
b_orth = np.array([-4.0, 3.0])
print(f'a@b_pos={a_pos@b_pos:.1f} (>0, angle<90)')
print(f'a@b_neg={a_pos@b_neg:.1f} (<0, angle>90)')
print(f'a@b_orth={a_pos@b_orth:.1f} (=0, perpendicular)')


---
## 5.2  几何本质 —— 点积 = |a||b|cosθ

a·b = |a||b|cosθ。推导出余弦相似度: cos_sim = a·b/(|a||b|) = cosθ，值域[-1,1]。

AI连接：推荐系统、语义搜索、人脸验证的基础度量。


In [ ]:
import numpy as np
def cosine_similarity(a, b):
    return np.dot(a,b)/(np.linalg.norm(a)*np.linalg.norm(b))
np.random.seed(42)
a = np.random.randn(3); b = np.random.randn(3)
dot_alg = np.dot(a, b)
cos_th = cosine_similarity(a, b)
dot_geo = np.linalg.norm(a)*np.linalg.norm(b)*cos_th
print(f'algebraic: {dot_alg:.6f}')
print(f'geometric: {dot_geo:.6f}')
print(f'match: {abs(dot_alg-dot_geo)<1e-10}')
print(f'cos_sim={cos_th:.4f}, angle={np.arccos(cos_th)*180/np.pi:.1f} deg')
v = np.array([3.0, 4.0])
print(f'v·v={v@v:.0f} cos=1.0 | v·(-v)={v@(-v):.0f} cos=-1.0 | v·[-4,3]={v@np.array([-4,3]):.0f} cos=0.0')


---
## 5.3  猜你喜欢 —— 用点积构建推荐系统

用户向量·电影向量 → 余弦相似度排序 → Top-K推荐。推荐系统=向量空间中的最近邻搜索。

AI连接：工业界用FAISS加速，底层还是点积。


In [ ]:
import numpy as np
movies = {
    '星际穿越': np.array([0.3,0.0,0.9,0.2,0.1]),
    '泰坦尼克': np.array([0.1,0.1,0.0,0.95,0.1]),
    '盗梦空间': np.array([0.6,0.0,0.7,0.1,0.5]),
    '喜剧之王': np.array([0.0,0.95,0.0,0.2,0.0]),
    '黑客帝国': np.array([0.8,0.0,0.6,0.1,0.3]),
    '爱乐之城': np.array([0.0,0.3,0.0,0.9,0.0]),
}
user = np.array([0.2,0.05,0.8,0.05,0.6])
def cs(a,b): return np.dot(a,b)/(np.linalg.norm(a)*np.linalg.norm(b)+1e-8)
scores = {n:cs(user,v) for n,v in movies.items()}
for n,s in sorted(scores.items(),key=lambda x:-x[1]):
    print(f'{n}: {s:.4f}')
top3 = sorted(scores.items(),key=lambda x:-x[1])[:3]
print(f'Top-3: {[t[0] for t in top3]}')


---
## 5.4  AI连接：Transformer中的Q·K = 点积规模化

scores = Q@K^T/sqrt(d_k)——矩阵乘法中每个元素就是一次点积。
类比：Q@K^T的每个点积=query想找什么和key有什么之间的余弦相似度。

AI连接：第29章Multi-Head Attention，第30章训练文本续写模型。


In [ ]:
import numpy as np
np.random.seed(42)
seq_len, d_k = 4, 8
Q = np.random.randn(seq_len, d_k)
K = np.random.randn(seq_len, d_k)
V = np.random.randn(seq_len, d_k)
scores = Q @ K.T  # (4,4) — each element is a dot product!
# Scale + softmax
scores_s = scores / np.sqrt(d_k)
scores_s = scores_s - scores_s.max(axis=-1, keepdims=True)
attn = np.exp(scores_s)
attn = attn / attn.sum(axis=-1, keepdims=True)
output = attn @ V  # (4,4) @ (4,8) = (4,8)
print(f'scores shape: {scores.shape}')
print(f'weights sum per row: {attn.sum(axis=1).round(3)}')
print(f'output shape: {output.shape}')
print(f'scores[0,1] == dot(Q[0],K[1]): {abs(scores[0,1]-np.dot(Q[0],K[1]))<1e-10}')


---
## 习题

1.（概念）点积为0/正/负分别代表什么几何关系？
2.（概念）余弦相似度和点积有什么区别？为什么推荐系统通常用余弦相似度？
3.（代码）对a=[1,2,3]和b=[4,-2,1]，用代数和几何两种方式计算点积并验证。
4.（代码）构建音乐推荐系统：8首歌×4维特征，2个用户，各推荐Top-3。


---
> **章末钩子**：点积让你衡量两个向量的相似度。但需要批量处理时——矩阵乘法一次性算出所有结果。
> 预览：下一章——**矩阵即变换**，从表格到空间扭曲。
> **提示**：完成后运行 Kernel -> Restart & Run All 验证。
